# 📊 Notebook 01: Exploratory Data Analysis (EDA)
## Intelligent Banking Transaction & Fraud Detection System

This notebook conducts in-depth exploratory analysis on the raw 300,000 transaction dataset, analyzing distributions, class imbalance, diurnal patterns, and fraud risk indicators.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

ROOT_DIR = Path.cwd().parent
sys.path.insert(0, str(ROOT_DIR))
from src.config import RAW_DATA_FILE

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 5)

### 1. Load Raw Dataset

In [ ]:
df = pd.read_csv(RAW_DATA_FILE, low_memory=False)
print(f"Dataset Dimensions: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

### 2. Class Imbalance Analysis
Fraudulent transactions are an extreme minority class in banking systems.

In [ ]:
fraud_counts = df["fraud_label"].value_counts()
fraud_rate = df["fraud_label"].mean() * 100
print(f"Legitimate Transactions (0): {fraud_counts[0]:,} ({100 - fraud_rate:.2f}%)")
print(f"Fraudulent Transactions (1): {fraud_counts[1]:,} ({fraud_rate:.2f}%)")

plt.figure(figsize=(6, 4))
sns.countplot(x="fraud_label", data=df, palette=["#059669", "#DC2626"])
plt.title("Severe Class Imbalance Distribution (2.1% Fraud)")
plt.xlabel("Fraud Status (0 = Legitimate, 1 = Fraud)")
plt.ylabel("Number of Transactions")
plt.show()

### 3. Transaction Amount Distribution (Log-Scale)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

sns.histplot(df["transaction_amount"], bins=50, kde=True, ax=ax1, color="#2563EB")
ax1.set_title("Raw Transaction Amounts ($)")

sns.histplot(np.log1p(df["transaction_amount"]), bins=50, kde=True, ax=ax2, color="#7C3AED")
ax2.set_title("Log1p Transformed Transaction Amounts")
plt.show()

### 4. Fraud Rate by Merchant Category

In [ ]:
merch_summary = df.groupby("merchant_category")["fraud_label"].agg(["count", "sum", "mean"]).reset_index()
merch_summary["mean"] = merch_summary["mean"] * 100
merch_summary = merch_summary.sort_values(by="mean", ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(x="mean", y="merchant_category", data=merch_summary, palette="Reds_r")
plt.title("Fraud Incidence Rate (%) by Merchant Category")
plt.xlabel("Fraud Rate (%)")
plt.ylabel("Merchant Category")
plt.show()

### 5. Diurnal Patterns: Fraud by Hour of the Day

In [ ]:
hourly = df.groupby("transaction_hour")["fraud_label"].mean() * 100
plt.figure(figsize=(10, 4))
hourly.plot(kind="bar", color="#DC2626")
plt.title("Fraud Probability (%) by Hour of Day (0 to 23)")
plt.xlabel("Hour of Day")
plt.ylabel("Fraud Rate (%)")
plt.show()